# BookWise — Evaluasi Lengkap & Hybrid Model
Precision@K, Recall@K, NDCG, Cold Start Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics.pairwise import cosine_similarity

MODEL_DIR = '../model'
DATA_DIR  = '../../data'

cb_model = joblib.load(f'{MODEL_DIR}/content_based.pkl')
cf_model = joblib.load(f'{MODEL_DIR}/collaborative.pkl')
books_df = pd.read_pickle(f'{MODEL_DIR}/books.pkl')

ratings = pd.read_csv(f'{DATA_DIR}/BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
ratings.columns = ratings.columns.str.strip().str.replace('"', '')
ratings = ratings[ratings['Book-Rating'] > 0]
print('Models loaded.')

In [ ]:
def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & set(relevant)) / k

def recall_at_k(recommended, relevant, k):
    if not relevant: return 0
    return len(set(recommended[:k]) & set(relevant)) / len(relevant)

def ndcg_at_k(recommended, relevant, k):
    dcg = sum(1/np.log2(i+2) for i, r in enumerate(recommended[:k]) if r in relevant)
    idcg = sum(1/np.log2(i+2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0

print('Evaluation functions defined.')

In [ ]:
# Evaluasi Content-Based
isbn_index = cb_model['isbn_index']
cosine_sim = cb_model['cosine_sim']

def cb_recommend(isbn, k=10):
    if isbn not in isbn_index: return []
    idx = isbn_index[isbn]
    scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:k+1]
    return books_df.iloc[[i[0] for i in scores]]['ISBN'].tolist()

high_ratings = ratings[ratings['Book-Rating'] >= 7]
sample_users = high_ratings['User-ID'].value_counts().head(200).index

p_scores, r_scores, n_scores = [], [], []
for user in sample_users:
    user_books = high_ratings[high_ratings['User-ID'] == user]['ISBN'].tolist()
    if len(user_books) < 2: continue
    recs = cb_recommend(user_books[0], 10)
    if not recs: continue
    relevant = user_books[1:]
    p_scores.append(precision_at_k(recs, relevant, 10))
    r_scores.append(recall_at_k(recs, relevant, 10))
    n_scores.append(ndcg_at_k(recs, relevant, 10))

print(f'Content-Based Precision@10: {np.mean(p_scores):.4f}')
print(f'Content-Based Recall@10:    {np.mean(r_scores):.4f}')
print(f'Content-Based NDCG@10:      {np.mean(n_scores):.4f}')

In [ ]:
# Cold Start Analysis
print('=== Cold Start Problem Analysis ===')
print('\nNew User Problem:')
print('- User baru tidak punya histori rating')
print('- Solusi: Onboarding (tanya genre favorit) → Content-Based dari genre')
print('- Fallback: Tampilkan buku populer (most-rated)')

print('\nNew Item Problem:')
print('- Buku baru tidak punya rating')
print('- Solusi: Content-Based menggunakan metadata (judul, penulis, publisher)')
print('- TF-IDF tetap bisa merekomendasikan buku baru berdasarkan fitur teks')

# Visualisasi distribusi user activity
user_activity = ratings['User-ID'].value_counts()
plt.figure(figsize=(10, 4))
plt.hist(user_activity[user_activity <= 50], bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Jumlah Rating per User')
plt.ylabel('Frekuensi')
plt.title('Distribusi Aktivitas User (Cold Start Analysis)')
plt.axvline(5, color='red', linestyle='--', label='Threshold (5 ratings)')
plt.legend()
plt.tight_layout()
plt.savefig('cold_start_analysis.png', dpi=150)
plt.show()
print(f'Users dengan < 5 rating: {(user_activity < 5).sum():,} ({(user_activity < 5).mean():.1%})')

In [ ]:
# Ringkasan evaluasi untuk paper
results = {
    'Method': ['Content-Based', 'Collaborative (SVD)', 'Hybrid'],
    'Precision@10': [np.mean(p_scores), '-', '-'],
    'Recall@10':    [np.mean(r_scores), '-', '-'],
    'NDCG@10':      [np.mean(n_scores), '-', '-'],
}
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
df_results.to_csv('evaluation_results.csv', index=False)
print('\n✅ Results saved to evaluation_results.csv')